In [2]:
import pandas as pd

df = pd.read_parquet(rf"/data/susanket/vlm/images.parquet")

In [ ]:
import io
import math
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image


def compute_image_entropy(img_bytes):
    """Calculates Shannon entropy for a JPEG/PNG byte string (0 to 8 bits)."""
    try:
        # Load byte stream into PIL and convert to grayscale
        img = Image.open(io.BytesIO(img_bytes)).convert("L")
        histogram = img.histogram()
        total_pixels = sum(histogram)

        # Compute Shannon Entropy
        entropy = 0.0
        for count in histogram:
            if count > 0:
                p = count / total_pixels
                entropy -= p * math.log2(p)
        return entropy
    except Exception:
        return None


# --- Notebook Plotting ---
def plot_sample_entropies(df, sample_size=6):
    # Sample random valid rows
    sampled_df = df.sample(n=min(sample_size, len(df))).copy()

    fig, axes = plt.subplots(
        1, len(sampled_df), figsize=(4 * len(sampled_df), 4)
    )
    if len(sampled_df) == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, sampled_df.iterrows()):
        img_dict = row["image"]
        byte_data = img_dict.get("bytes") if isinstance(img_dict, dict) else None

        if byte_data:
            entropy = compute_image_entropy(byte_data)
            img = Image.open(io.BytesIO(byte_data))

            ax.imshow(img)
            ax.set_title(f"Entropy: {entropy:.3f}", fontsize=12)
        else:
            ax.set_title("Invalid Image Data")

        ax.axis("off")

    plt.tight_layout()
    plt.show()


# Run visualization
plot_sample_entropies(df, sample_size=5)